# Rider Risk Profiling — K-Means Clustering

Cluster a **fleet of synthetic trips** into three risk tiers:
**Low / Medium / High Risk** — no labels required.

Each trip is condensed into 16 scalar features (mean magnitude, %
sharp turns, fall count, smooth score, road severity, etc.).
K-Means then groups trips by riding behaviour.

Real-world use: insurance scoring, fleet safety monitoring, maintenance scheduling.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent.parent
sys.path.insert(0, str(ROOT / 'ml'))
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.decomposition import PCA

from scooter_imu.data.preprocessor             import preprocess
from scooter_imu.algorithms.turn_indicator     import TurnIndicator
from scooter_imu.algorithms.fall_detection     import FallDetector
from scooter_imu.algorithms.road_quality       import RoadQualityAnalyzer
from scooter_imu.algorithms.riding_style       import RidingStyleScorer
from scooter_imu.algorithms.lane_classifier    import LaneClassifier

from risk_profiler import RiskProfiler, aggregate_trip, RISK_LABELS, RISK_COLORS
import generate_trip_data as gtd

print('Libraries loaded.')

Libraries loaded.


## 1. Generate Fleet Data (30 Synthetic Trips)

In [2]:
def run_algorithms(df):
    for algo in [TurnIndicator(), FallDetector(), RoadQualityAnalyzer(),
                 RidingStyleScorer(), LaneClassifier()]:
        df = algo.predict(df)
    return df

N_TRIPS = 30
feature_rows = []
trip_dfs     = []

for seed in range(N_TRIPS):
    df = run_algorithms(preprocess(gtd.generate_trip_df(seed)))
    feature_rows.append(aggregate_trip(df))
    trip_dfs.append(df)

fleet_df = pd.DataFrame(feature_rows)
print(f'Fleet feature matrix: {fleet_df.shape}')
fleet_df.describe().round(3)

Fleet feature matrix: (30, 16)


,mean_magnitude,max_magnitude,p95_magnitude,pct_left_turn,pct_right_turn,pct_sharp_turn,n_falls,pct_fall_duration,mean_road_severity,pct_severe_road,mean_smooth_score,min_smooth_score,pct_low_score,pct_lane_change,pct_straight,duration_s
count,30.000,30.000,30.000,30.000,30.000,30.000,30.0,30.000,30.000,30.000,30.000,30.000,30.000,30.000,30.000,30.00
mean,0.138,5.099,0.489,0.086,0.119,0.014,1.0,0.001,0.184,0.058,78.034,43.060,0.047,0.030,0.812,599.98
std,0.000,0.121,0.004,0.000,0.000,0.000,0.0,0.000,0.001,0.001,0.266,7.276,0.006,0.001,0.000,0.00
min,0.137,4.844,0.481,0.085,0.119,0.014,1.0,0.001,0.183,0.055,77.069,5.600,0.039,0.030,0.812,599.98
25%,0.138,5.026,0.486,0.086,0.119,0.014,1.0,0.001,0.184,0.057,77.938,44.000,0.045,0.030,0.812,599.98
50%,0.138,5.088,0.489,0.086,0.119,0.014,1.0,0.001,0.185,0.058,78.066,44.550,0.046,0.030,0.812,599.98
75%,0.138,5.174,0.491,0.086,0.119,0.014,1.0,0.001,0.185,0.058,78.215,45.075,0.046,0.031,0.812,599.98
max,0.139,5.382,0.496,0.086,0.120,0.014,1.0,0.001,0.186,0.059,78.412,46.500,0.067,0.032,0.812,599.98


## 2. K-Means Clustering (3 Risk Tiers)

In [3]:
rp = RiskProfiler(n_clusters=3)
rp.fit(feature_rows)
labels = rp.predict(feature_rows)

fleet_df['risk_cluster'] = labels
fleet_df['risk_label']   = [RISK_LABELS[l] for l in labels]

print('Cluster distribution:')
print(fleet_df['risk_label'].value_counts().to_string())

Cluster distribution:
risk_label
Medium Risk    20
High Risk       9
Low Risk        1


## 3. PCA Scatter — Cluster Visualisation

In [4]:
from sklearn.preprocessing import StandardScaler

feat_cols = [c for c in fleet_df.columns if c not in ('risk_cluster', 'risk_label')]
X_scaled  = StandardScaler().fit_transform(fleet_df[feat_cols])
pca       = PCA(n_components=2, random_state=42)
coords    = pca.fit_transform(X_scaled)

fig = go.Figure()
for cluster_id, label in RISK_LABELS.items():
    mask = labels == cluster_id
    fig.add_trace(go.Scatter(
        x=coords[mask, 0], y=coords[mask, 1],
        mode='markers+text',
        marker=dict(color=RISK_COLORS[cluster_id], size=12,
                    line=dict(color='white', width=1)),
        text=[str(i) for i in np.where(mask)[0]],
        textposition='top center',
        name=label,
    ))

fig.update_layout(
    title=f'Trip Clusters — PCA Projection  (explained variance: {pca.explained_variance_ratio_.sum():.1%})',
    xaxis_title=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
    yaxis_title=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    height=480, template='plotly_white',
)
fig.show()

## 4. Cluster Centroid Profiles

In [5]:
centers = rp.cluster_centers_df()
print('Cluster centroids (original feature scale):')
display(centers.round(3).T)

Cluster centroids (original feature scale):


,Low Risk,Medium Risk,High Risk
mean_magnitude,0.138,0.138,0.139
max_magnitude,4.999,5.064,5.189
p95_magnitude,0.485,0.488,0.493
pct_left_turn,0.086,0.086,0.086
pct_right_turn,0.119,0.119,0.119
pct_sharp_turn,0.014,0.014,0.014
n_falls,1.000,1.000,1.000
pct_fall_duration,0.001,0.001,0.001
mean_road_severity,0.184,0.184,0.185
pct_severe_road,0.058,0.057,0.058


## 5. Radar Chart — Risk Tier Comparison

In [6]:
# Select a subset of interpretable dimensions for the radar
radar_dims = [
    'mean_magnitude', 'pct_sharp_turn', 'n_falls',
    'mean_road_severity', 'min_smooth_score', 'pct_lane_change'
]
radar_labels = [
    'Avg Intensity', 'Sharp Turns %', 'Fall Events',
    'Road Severity', 'Min Smooth Score', 'Lane Changes %'
]

# Normalise each dimension to [0, 1] for comparability
radar_df = centers[radar_dims].copy()
radar_df = (radar_df - radar_df.min()) / (radar_df.max() - radar_df.min() + 1e-9)

fig = go.Figure()
for cluster_id, row in radar_df.iterrows():
    vals = list(row) + [row.iloc[0]]   # close the polygon
    cats = radar_labels + [radar_labels[0]]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=cats, fill='toself',
        name=cluster_id,
        line=dict(color=RISK_COLORS[list(RISK_LABELS.values()).index(cluster_id)]),
    ))

fig.update_layout(
    title='Risk Tier Profiles (normalised)',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    height=480, template='plotly_white',
)
fig.show()

## 6. Score a New Trip

In [7]:
# Use the original data file as a new unseen trip
from scooter_imu.data.loader import load_raw

new_df    = run_algorithms(preprocess(load_raw(ROOT / 'data/raw/ScooterIMUData.csv')))
cluster, label = rp.predict_one(new_df)
color  = RISK_COLORS[cluster]

fig = go.Figure(go.Indicator(
    mode='number+gauge',
    value=cluster,
    title={'text': f'Trip Risk Assessment: <b>{label}</b>'},
    gauge=dict(
        axis=dict(range=[0, 2], tickvals=[0, 1, 2],
                  ticktext=['Low', 'Medium', 'High']),
        bar=dict(color=color),
        steps=[
            dict(range=[0, 0.5],  color='#d4f5e0'),
            dict(range=[0.5, 1.5],color='#fdecc8'),
            dict(range=[1.5, 2],  color='#fad0cc'),
        ],
    ),
))
fig.update_layout(height=280, paper_bgcolor='#f8fbff')
fig.show()

## 7. Save Model

In [8]:
rp.save(ROOT / 'ml/models/risk_kmeans.joblib')
print('Model saved. Reload with:  RiskProfiler.load()')

RiskProfiler saved → /Users/pulkit/Desktop/test/Automatic-Indicator-using-Gyroscope-data/ml/models/risk_kmeans.joblib
Model saved. Reload with:  RiskProfiler.load()
